# 🦁 WAVIS v2 — Google Colab Training Notebook
## Train EfficientNet-B3 on wildlife sounds → 95%+ accuracy

**Steps:**
1. Run all cells top to bottom
2. Training takes ~45-60 min on Colab T4 GPU
3. Download `wavis_v2_best.pth` at the end
4. Put it in your local `models/` folder

**Runtime:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────────────────────
import torch
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — change runtime!"}')
print(f'CUDA: {torch.version.cuda}')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────────
!pip install -q timm audiomentations librosa soundfile plotly pyyaml
print('✅ Packages installed')

In [ ]:
# ── Cell 3: Upload your WAVIS project zip ─────────────────────────────────────
# Upload wavis_v2_project.zip using the Files panel on the left
# Then run this cell to extract it

import zipfile, os

# If you uploaded the zip:
if os.path.exists('/content/wavis_v2_project.zip'):
    with zipfile.ZipFile('/content/wavis_v2_project.zip') as z:
        z.extractall('/content')
    print('✅ Project extracted')
else:
    # OR: clone from your GitHub if you pushed it
    # !git clone https://github.com/YOUR_USERNAME/wavis /content/wavis_v2
    print('⚠️ Upload wavis_v2_project.zip first via Files panel')

os.chdir('/content/wavis_v2')
print(f'Working dir: {os.getcwd()}')
!ls

In [ ]:
# ── Cell 4: Download Dataset ───────────────────────────────────────────────────
# Option A: Quick dataset (ESC-50 + Xeno-Canto, ~700MB, ~70-80% accuracy)
!python download_dataset.py --quick

# Option B: Full BirdCLEF (requires Kaggle API setup, ~38GB, 95%+ accuracy)
# from google.colab import files
# files.upload()  # upload your kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !python download_dataset.py --full

In [ ]:
# ── Cell 5: Train ─────────────────────────────────────────────────────────────
# EfficientNet-B3, 50 epochs, AMP enabled
# Expected: 80-90% on quick dataset, 95%+ on full BirdCLEF

!python train.py --epochs 50

# For even better accuracy:
# !python train.py --epochs 80 --backbone efficientnet_b5

In [ ]:
# ── Cell 6: Plot training curves ──────────────────────────────────────────────
import json, matplotlib.pyplot as plt

with open('models/training_history.json') as f:
    hist = json.load(f)['history']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor='#04100a')

for ax in [ax1, ax2]:
    ax.set_facecolor('#071a10')
    for spine in ax.spines.values():
        spine.set_color('#1a3a25')
    ax.tick_params(colors='#86efac')

ax1.plot(hist['train_acc'], color='#4ade80', label='Train Acc', linewidth=2)
ax1.plot(hist['val_acc'],   color='#fb923c', label='Val Acc',   linewidth=2)
ax1.plot(hist['val_top5'],  color='#60a5fa', label='Val Top-5', linewidth=2, linestyle='--')
ax1.set_title('Accuracy', color='#86efac')
ax1.set_xlabel('Epoch', color='#86efac')
ax1.set_ylabel('Accuracy (%)', color='#86efac')
ax1.legend(facecolor='#071a10', labelcolor='#86efac')
ax1.axhline(95, color='#4ade80', linestyle=':', alpha=0.4, label='95% target')

ax2.plot(hist['train_loss'], color='#4ade80', label='Train Loss', linewidth=2)
ax2.plot(hist['val_loss'],   color='#fb923c', label='Val Loss',   linewidth=2)
ax2.set_title('Loss', color='#86efac')
ax2.set_xlabel('Epoch', color='#86efac')
ax2.set_ylabel('Loss', color='#86efac')
ax2.legend(facecolor='#071a10', labelcolor='#86efac')

plt.tight_layout()
plt.savefig('models/training_curves.png', dpi=120, bbox_inches='tight', facecolor='#04100a')
plt.show()
print('Training curves saved!')

In [ ]:
# ── Cell 7: Download model ────────────────────────────────────────────────────
from google.colab import files

# Download the trained model
files.download('models/wavis_v2_best.pth')

# Also download class maps
files.download('data/class_map.json')
files.download('data/display_names.json')

print('✅ Downloaded! Place wavis_v2_best.pth in your local models/ folder.')
print('   Place class_map.json and display_names.json in your local data/ folder.')
print('   Then run: streamlit run app.py')